# AI-LUNG Data Sanity + Preprocessing Notebook
This notebook runs the planned end-to-end workflow in VS Code and remains compatible with Google Colab.

## 1) Environment Setup and Imports
- Verifies package imports
- Configures plotting/logging
- Keeps output deterministic where possible

In [ ]:
import os
import sys
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure src/ is importable when running notebook directly
repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from ailung.dataset import discover_ct_series
from ailung.preprocess import build_volume, hu_clip_normalize, resample_isotropic
from ailung.annotations import parse_lidc_xml

plt.rcParams['figure.figsize'] = (8, 8)
print('Imports OK')

## 2) Project Configuration
Define paths once so this notebook works both locally and in Colab.

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Toggle this when running in Colab
RUN_ENV = 'local'  # 'local' or 'colab'

if RUN_ENV == 'local':
    DATASET_ROOT = Path('d:/AI_LUNG/manifest-1600709154662')
    METADATA_CSV = DATASET_ROOT / 'metadata.csv'
    EXPORT_DIR = Path('d:/AI_LUNG/outputs/sanity')
else:
    # Example Colab paths after mounting Google Drive
    DATASET_ROOT = Path('/content/drive/MyDrive/AI_LUNG/manifest-1600709154662')
    METADATA_CSV = DATASET_ROOT / 'metadata.csv'
    EXPORT_DIR = Path('/content/drive/MyDrive/AI_LUNG/outputs/sanity')

SAMPLE_INDEX = 0
TARGET_SPACING = 1.0
HU_MIN, HU_MAX = -1000, 400

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
print({'dataset_root': str(DATASET_ROOT), 'metadata_csv': str(METADATA_CSV), 'export_dir': str(EXPORT_DIR)})

## 3) Load Inputs
Load metadata, discover CT series, and inspect a selected case.

In [ ]:
metadata = pd.read_csv(METADATA_CSV)
print('metadata rows:', len(metadata))
print(metadata[['Subject ID', 'Modality', 'Number of Images']].head(5))

series_list = discover_ct_series(DATASET_ROOT, METADATA_CSV)
print('ct series discovered:', len(series_list))

selected = series_list[SAMPLE_INDEX]
print('selected subject:', selected.subject_id)
print('selected path:', selected.file_location)
print('expected images (metadata):', selected.number_of_images)

## 4) Validate and Preprocess Data
Build HU volume from DICOM, apply clipping/normalization, and enforce expected shapes/ranges.

In [ ]:
volume_hu, spacing = build_volume(selected.file_location)
volume_norm = hu_clip_normalize(volume_hu, hu_min=HU_MIN, hu_max=HU_MAX)

assert volume_hu.ndim == 3, 'Expected 3D HU volume'
assert volume_norm.ndim == 3, 'Expected 3D normalized volume'
assert np.isfinite(volume_hu).all(), 'HU volume contains non-finite values'
assert (volume_norm.min() >= 0.0) and (volume_norm.max() <= 1.0), 'Normalization out of [0,1] range'

print('raw shape (z,y,x):', volume_hu.shape)
print('spacing (z,y,x):', spacing)
print('HU min/max:', float(volume_hu.min()), float(volume_hu.max()))
print('Norm min/max:', float(volume_norm.min()), float(volume_norm.max()))

## 5) Implement Core Workflow
Resample to isotropic spacing, parse XML annotations, and keep intermediate outputs for reuse.

In [ ]:
volume_iso, iso_spacing = resample_isotropic(volume_norm, spacing, target_spacing=TARGET_SPACING)

xml_files = sorted(Path(selected.file_location).glob('*.xml'))
annotations = parse_lidc_xml(xml_files[0]) if xml_files else {'nodules': []}

workflow_outputs = {
    'subject_id': selected.subject_id,
    'series_uid': selected.series_uid,
    'raw_shape': tuple(map(int, volume_hu.shape)),
    'raw_spacing': tuple(map(float, spacing)),
    'iso_shape': tuple(map(int, volume_iso.shape)),
    'iso_spacing': tuple(map(float, iso_spacing)),
    'nodule_entries': int(len(annotations.get('nodules', []))),
}

print(json.dumps(workflow_outputs, indent=2))

## 6) Run Checks and Metrics
Compute basic diagnostics and visualize a representative middle slice.

In [ ]:
raw_voxels = int(np.prod(volume_hu.shape))
iso_voxels = int(np.prod(volume_iso.shape))

print('raw voxels:', raw_voxels)
print('isotropic voxels:', iso_voxels)
print('nodule entries:', len(annotations.get('nodules', [])))

mid = volume_norm.shape[0] // 2
plt.imshow(volume_norm[mid], cmap='gray')
plt.title(f'Middle normalized slice: z={mid}')
plt.axis('off')
plt.show()

## 7) Add Unit-Test Cells
Notebook-friendly asserts for critical functions.

In [ ]:
def _run_quick_tests():
    assert len(series_list) > 0
    assert volume_hu.shape[0] > 16
    assert volume_hu.shape[1] > 64 and volume_hu.shape[2] > 64
    assert volume_norm.dtype == np.float32
    assert volume_iso.dtype == np.float32
    assert len(iso_spacing) == 3
    assert all(abs(s - TARGET_SPACING) < 1e-6 for s in iso_spacing)

_run_quick_tests()
print('Quick tests passed ✅')

## 8) Export Results
Save lightweight artifacts and reproducibility metadata.

In [ ]:
np.save(EXPORT_DIR / 'sample_volume_norm.npy', volume_norm)
np.save(EXPORT_DIR / 'sample_volume_iso.npy', volume_iso)

summary = {
    'seed': SEED,
    'run_env': RUN_ENV,
    'dataset_root': str(DATASET_ROOT),
    'subject_id': selected.subject_id,
    'series_uid': selected.series_uid,
    'raw_shape': tuple(map(int, volume_hu.shape)),
    'raw_spacing': tuple(map(float, spacing)),
    'iso_shape': tuple(map(int, volume_iso.shape)),
    'iso_spacing': tuple(map(float, iso_spacing)),
    'nodule_entries': int(len(annotations.get('nodules', []))),
}

with open(EXPORT_DIR / 'sanity_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print('Exported:')
print('-', EXPORT_DIR / 'sample_volume_norm.npy')
print('-', EXPORT_DIR / 'sample_volume_iso.npy')
print('-', EXPORT_DIR / 'sanity_summary.json')